# Copilot Cost Consumption — Ingester (Fabric)

Lands the **Microsoft 365 Admin Center → Copilot → Cost management → Consumption** per-user CSV export into a Lakehouse Delta table that the **AI Business Value Dashboard** reads for **Cowork / WorkIQ / Other** credit attribution by department.

Why this exists: the Power Platform `EntitlementConsumption*_MCSMessages_*` exports (landed by `Copilot_Credit_Consumption_Ingester`) meter Copilot **Studio message** credits per agent/user/tenant. They do **not** contain **Cowork** or **WorkIQ** consumption. This MAC Cost management export is the only customer-pullable place those surface-split credits appear. This source is **additive** to `credit_consumption_*` — not a replacement; the two answer different questions.

```
M365 Admin Center  ─(export)▶  Power Automate flow  ▶  Lakehouse Files/cost_consumption/*.csv  ▶  THIS notebook  ▶  Delta
```

| Source CSV | → Delta table | Grain |
| --- | --- | --- |
| MAC Cost management → Consumption export (per-user) | `dbo.copilot_cost_consumption` | one row per user (snapshot) |

**Source contract** (header → sanitised name): `User Principal Name`→`User_Principal_Name` (join key), `Cowork Credits`→`Cowork_Credits`, `WorkIQ Credits`→`WorkIQ_Credits`, `Other Credits`→`Other_Credits`, `Last Activity Date`→`Last_Activity_Date`. The ingester adds `Total_Credits` (= sum, nulls→0) plus `SourceFile` / `LoadDate` lineage.

**Graceful by design** — optional. A customer who never sends the export simply gets an empty, correctly-named table, and the PBIP's `Enable_CostConsumption` toggle keeps those pages dormant. A missing file is skipped with a warning, never an error (unless `STRICT`).

**No app registration / Graph permission required** — this notebook only reads files already in the Lakehouse.

## 1. Configuration

Tag this cell as the pipeline **`parameters`** cell (cell ⋯ menu → *Toggle parameter cell*) so the adoption pipeline can override `WRITE_MODE` per run. The output table is schema-qualified (`dbo.`) to work with both schema-enabled and legacy Lakehouses.

In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# Folder in the attached Lakehouse where the Power Automate flow lands the CSV(s).
# MUST match TargetFolder in the Copilot_CostConsumption_*_to_OneLake flows.
SOURCE_DIR   = 'Files/cost_consumption'

# The MAC Cost management export filename is not fixed, so we glob every .csv in the folder
# and union them (a customer may drop one combined export or several).
SOURCE_GLOB  = '*'
OUTPUT_TABLE = 'dbo.copilot_cost_consumption'

WRITE_MODE   = 'overwrite'   # 'overwrite' = full snapshot (this is a per-user point-in-time export);
                             # 'append'    = keep history (a LoadDate column is always added either way)

ADD_TOTAL    = True          # add Total_Credits = Cowork + WorkIQ + Other (nulls -> 0)
ADD_LINEAGE  = True          # add SourceFile + LoadDate columns to every row
STRICT       = False         # False = skip a missing/empty export with a warning; True = raise

## 2. Helpers

`_list_matches` resolves a glob against the Lakehouse `Files/` area (via the local `/lakehouse/default/` mount). `_read_csv` parses with multiline + quote-escape so any free-text survives. `_sanitize` makes column names Delta-safe (`[ ,;{}()\n\t=/-]` → `_`, BOM stripped, casing preserved) while the PBIP keeps reading the contract names through Delta column-mapping. `_typed` casts the three credit columns to double, parses `Last_Activity_Date` (ISO or US `M/d/yyyy`), and derives `Total_Credits`.

In [ ]:
import os, glob, re, datetime
from pyspark.sql import functions as F

_LOAD_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

# Local POSIX mount for the attached Lakehouse: 'Files/x' -> '/lakehouse/default/Files/x'
def _local(path):
    return path if path.startswith('/') else f'/lakehouse/default/{path}'

def _list_matches(pattern):
    """Return Spark-readable 'Files/...' paths matching a case-insensitive glob (.csv only)."""
    base = _local(SOURCE_DIR)
    if not os.path.isdir(base):
        return []
    rx = re.compile('^' + re.escape(pattern).replace('\\*', '.*') + '$', re.IGNORECASE)
    hits = [f for f in os.listdir(base) if rx.match(f) and f.lower().endswith('.csv')]
    return [f'{SOURCE_DIR}/{f}' for f in sorted(hits)]

def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

_INVALID = re.compile(r'[ ,;{}()\n\t=/-]')
def _sanitize(df):
    return df.toDF(*[_INVALID.sub('_', c.lstrip('\ufeff')) for c in df.columns])

_CREDIT_COLS = ['Cowork_Credits', 'WorkIQ_Credits', 'Other_Credits']
def _typed(df):
    """Cast credit columns to double, parse Last_Activity_Date, derive Total_Credits.
    Defensive: only touches columns that exist so a partial export can't break the run."""
    cols = set(df.columns)
    out = df
    for c in _CREDIT_COLS:
        if c in cols:
            out = out.withColumn(c, F.col(c).cast('double'))   # blank -> null automatically
    if 'Last_Activity_Date' in cols:
        out = out.withColumn('Last_Activity_Date', F.coalesce(
            F.to_date('Last_Activity_Date', 'yyyy-MM-dd'),
            F.to_date('Last_Activity_Date', 'M/d/yyyy'),
            F.to_date('Last_Activity_Date', "yyyy-MM-dd'T'HH:mm:ss"),
            F.to_date('Last_Activity_Date')))
    if ADD_TOTAL:
        comps = [F.coalesce(F.col(c).cast('double'), F.lit(0.0)) for c in _CREDIT_COLS if c in cols]
        if comps:
            total = comps[0]
            for x in comps[1:]:
                total = total + x
            out = out.withColumn('Total_Credits', total)
        else:
            out = out.withColumn('Total_Credits', F.lit(None).cast('double'))
    return out

print(f'Load timestamp: {_LOAD_TS}')
print(f'Source folder : {SOURCE_DIR}  (exists: {os.path.isdir(_local(SOURCE_DIR))})')

## 3. Ingest → Delta

Globs every `.csv` in `Files/cost_consumption`, unions them by name (so multiple drops merge), sanitises, types, adds lineage, and writes `dbo.copilot_cost_consumption`. A missing/empty export writes an **empty, correctly-named** table so the PBIP query never errors with *"the column … wasn't found"*.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

# Canonical (sanitised) contract — used to build an empty table when the export is absent,
# so downstream PBIP columns always resolve.
BASE_COLS = ['User_Principal_Name', 'Cowork_Credits', 'WorkIQ_Credits', 'Other_Credits', 'Last_Activity_Date']

def _empty():
    fields = [
        StructField('User_Principal_Name', StringType(), True),
        StructField('Cowork_Credits',      DoubleType(), True),
        StructField('WorkIQ_Credits',      DoubleType(), True),
        StructField('Other_Credits',       DoubleType(), True),
        StructField('Last_Activity_Date',  DateType(),   True),
    ]
    if ADD_TOTAL:
        fields.append(StructField('Total_Credits', DoubleType(), True))
    if ADD_LINEAGE:
        fields.append(StructField('SourceFile', StringType(), True))
        fields.append(StructField('LoadDate',   StringType(), True))
    return spark.createDataFrame([], StructType(fields))

def _write(df, table):
    (df.write.mode(WRITE_MODE)
        .option('overwriteSchema', 'true')
        .option('delta.columnMapping.mode', 'name')
        .option('delta.minReaderVersion', '2')
        .option('delta.minWriterVersion', '5')
        .format('delta').saveAsTable(table))

matches = _list_matches(SOURCE_GLOB)
if not matches:
    msg = f'no .csv matched "{SOURCE_GLOB}" in {SOURCE_DIR}'
    if STRICT:
        raise FileNotFoundError(f'{OUTPUT_TABLE}: {msg}')
    print(f'⚠  {OUTPUT_TABLE} — {msg}; writing EMPTY table so the PBIP still resolves.')
    _write(_empty(), OUTPUT_TABLE)
    rows = 0
else:
    frames = []
    for path in matches:
        d = _typed(_sanitize(_read_csv(path)))
        if ADD_LINEAGE:
            d = d.withColumn('SourceFile', F.lit(os.path.basename(path))) \
                 .withColumn('LoadDate',   F.lit(_LOAD_TS))
        frames.append(d)

    df = frames[0]
    for d in frames[1:]:
        df = df.unionByName(d, allowMissingColumns=True)

    rows = df.count()
    _write(df, OUTPUT_TABLE)
    print(f'✓  {OUTPUT_TABLE} — {len(matches)} file(s), {rows:,} rows -> written ({WRITE_MODE})')

print('\nDone. cost-consumption Delta table written. Refresh the PBIP / Direct Lake model to pick it up.')

## 4. Verify

Prints user count, Cowork / WorkIQ / Other credit totals, and the surface mix % so you can sanity-check the export landed and parsed (dates not all null, credits sensible).

In [ ]:
def _num(df, col):
    return df.select(F.sum(F.coalesce(F.col(col).cast('double'), F.lit(0.0)))).collect()[0][0] or 0.0

t = spark.table(OUTPUT_TABLE)
n = t.count()
users = t.select(F.countDistinct('User_Principal_Name')).collect()[0][0] if n else 0
print(f'{OUTPUT_TABLE}: {n:,} row(s), {users:,} distinct user(s)')

if n:
    cw = _num(t, 'Cowork_Credits')
    wq = _num(t, 'WorkIQ_Credits')
    ot = _num(t, 'Other_Credits')
    tot = cw + wq + ot
    dated = t.filter(F.col('Last_Activity_Date').isNotNull()).count()
    print(f'  Cowork : {cw:,.2f}')
    print(f'  WorkIQ : {wq:,.2f}')
    print(f'  Other  : {ot:,.2f}')
    print(f'  TOTAL  : {tot:,.2f}')
    if tot:
        print(f'  surface mix — Cowork {cw/tot:.1%} | WorkIQ {wq/tot:.1%} | Other {ot/tot:.1%}')
    print(f'  Last_Activity_Date parsed (non-null): {dated:,}/{n:,}')
else:
    print('  (empty table — no export landed yet; PBIP pages stay dormant via Enable_CostConsumption.)')

---
**Connect the PBIP**: this table feeds the semantic-model query `copilot_cost_consumption`, gated by the `Enable_CostConsumption` parameter. Leave `Enable_CostConsumption = "Exclude"` for customers who don't supply the export and the Cost Consumption visuals stay dormant. `User_Principal_Name` joins the org table (`Chat + Agent Org Data[PersonId]`) for department / chargeback attribution; `Last_Activity_Date` joins `Calendar[Date]`. This source is **additive** to the PPAC `credit_consumption_*` tables.